In [1]:
import numpy as np
from hamilton import driver
from hamilton.execution import executors

import rayfish
import rayfish.quantum_fisher_information as qfi
from api.backend import create_dataframe_from_numpy_array
from api.visualize import PointSourceDistribution, StatisticalBounds
from rayfish.classical_fisher_information import (
    incoherent as classical_incoherent_sources,
)
from rayfish.classical_fisher_information.measurement_basis import (
    hermite_gaussian_basis,
)
from rayfish.parameter_catalogue import distance_2_sources_x_only
from rayfish.quantum_fisher_information import (
    incoherent_sources as qfi_incoherent_sources,
)

In [2]:
conf = {
    # "light_type": "incoherent",
    # "estimation_parameter": "distance_2_sources",
    "simulation_dimension": 1,
}

input_ = {
    "wave_vector": 2.0,
    "rayleigh_length": 1.0,
    "number_of_grids": 20,
    "maximal_length": 4.0,
    "minimal_spacing": 0.1,
}

In [3]:
quantum_dr = (
    driver.Builder()
    .with_modules(rayfish, qfi, qfi_incoherent_sources, distance_2_sources_x_only)
    .with_config(config=conf)
    .enable_dynamic_execution(allow_experimental_mode=True)
    .with_local_executor(executors.SynchronousLocalTaskExecutor())
).build()

classical_dr = (
    driver.Builder()
    .with_modules(
        rayfish,
        classical_incoherent_sources,
        distance_2_sources_x_only,
        hermite_gaussian_basis,
    )
    .with_config(config=conf)
).build()

In [4]:
quantum_result = quantum_dr.execute(
    inputs=input_,
    final_vars=[
        "covariance_matrix_for_entire_distribution",
        "number_of_point_sources",
        "spacing",
        "coordinates",
    ],
)

classical_result = classical_dr.execute(
    inputs={"number_of_modes": 10, **input_},
    final_vars=[
        "covariance_matrix_for_entire_distribution",
        "number_of_point_sources",
        "spacing",
        "coordinates",
    ],
)

In [5]:
grid_spacing = PointSourceDistribution(
    number_of_point_sources=quantum_result["number_of_point_sources"],
    grid_spacing=quantum_result["spacing"],
    coordinates=quantum_result["coordinates"],
)

In [6]:
temp_df = create_dataframe_from_numpy_array(
    grid_spacing.grid_spacing,
    quantum_result["covariance_matrix_for_entire_distribution"],
)
qfi_bounds = StatisticalBounds("quantum", temp_df.copy())

qfi_bounds.generate_figure()

In [7]:
bounds = []
for i in range(4):
    temp_df = create_dataframe_from_numpy_array(
        grid_spacing.grid_spacing,
        classical_result["covariance_matrix_for_entire_distribution"][:, i, :, :],
    )
    bounds.append(StatisticalBounds(str(i), temp_df.copy()))


temp_df = create_dataframe_from_numpy_array(
    grid_spacing.grid_spacing,
    np.sum(classical_result["covariance_matrix_for_entire_distribution"], axis=1),
)
bounds.append(StatisticalBounds("sum", temp_df.copy()))

bounds[-1].generate_figure()